### Setup

In [1]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="2"
import torch
import random
import numpy as np
from datetime import datetime
import os
import json
import shutil
from datasets import Dataset, DatasetDict
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance

from huggingface_hub import login
from diffusers import DDPMScheduler, StableDiffusionPipeline, StableDiffusionPipeline
from torchmetrics.image.fid import FrechetInceptionDistance 
from metric import *

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_SEED = 42

# TRAIN PARAMETER
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 1e-04
TRAIN_DATA_SIZE = 1800
TEST_DATA_SIZE = 200

# PATH
CONFIG_PATH = '../config.json'
## DATA
TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_KFashion' 
TEST_LABEL_FOLDER = '../Data/Total/Test_Label_CLIP_Summarize' 
# TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_Nike' 
# TEST_LABEL_FOLDER = '../Data/Total/Test_Label_Nike' 
# TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_Zara' 
# TEST_LABEL_FOLDER = '../Data/Total/Test_Label_Zara' 
TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0]))
TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0]))

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"

# HUGGINGFACE
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
HUGGING_FACE_TOKEN = config.get("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

2025-03-05 15:05:24.006820: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-05 15:05:24.034750: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-05 15:05:24.507990: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Instructions for updating:
non-resource variables are not supported in the long term
device :  cuda
package version :  2.4.1+cu124 4.46.2 3.0.1 0.33.0.dev0 0.7.0


### Load Model

In [2]:
# load orginal model 
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250303_224850'
pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16)
pipe.to("cuda")
# load fine-tunined model 
pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True) 
pipe.to("cuda")

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용
vae = pipe.vae.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용
unet = pipe.unet.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/gayeon38/anaconda3/envs/vlm-env/lib/python3.8/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


### Preprocess Data

In [3]:
# 주어진 데이터 샘플에서 text 컬럼을 토크나이징하여 토큰 ID tensor를 반환하는 함수.
def tokenize_captions(examples, caption_column='text', is_train=True):
    captions = []
    for caption in examples[caption_column]:
        # 캡션이 하나인 경우
        if isinstance(caption, str):
            captions.append(caption)
        # 캡션이 하나 이상인 경우 아무거나 하나 선택
        elif isinstance(caption, (list, np.ndarray)):
            # take a random caption if there are multiple
            captions.append(random.choice(caption) if is_train else caption[0])
        else:
            raise ValueError(
                f"Caption column `{caption_column}` should contain either strings or lists of strings."
            )
    inputs = tokenizer(
        captions, max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt"
    )

    return inputs.input_ids

# 이미지 데이터를 학습용으로 전처리하는 변환 파이프라인
transforms = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(IMG_SIZE) if True else transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip() if True else transforms.Lambda(lambda x: x),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]), # (0,1)->(-1,1)
    ]
)

# 이미지 데이터를 RGB로 변환 후 전처리하고, text 컬럼을 토크나이징하여 학습 데이터셋을 구성하는 함수.
def preprocess_data(examples, image_column='image'):
    images = [image.convert("RGB") for image in examples[image_column]]
    # 이미지 전처리
    examples["pixel_values"] = [transforms(image) for image in images]
    # 텍스트 전처리
    examples["input_ids"] = tokenize_captions(examples)
    return examples

# 배치 단위로 이미지와 토큰 ID를 스택하여 PyTorch 텐서 형태로 변환하는 함수.
def collate_fn(examples):
    # (C, H, W), ..., (C, H, W) -> stack -> (N, C, H, W): N으로 스택
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    # tensor.contiguous()와 동일
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()
    # {(77,), ..., (77,)}_N개 -> stack -> (N, 77)
    input_ids = torch.stack([example["input_ids"] for example in examples])

    return {"pixel_values": pixel_values, "input_ids": input_ids}


In [4]:
data = []
for image_file, label_file in zip(TEST_IMAGE_FILE, TEST_LABEL_FILE):
    image_path = os.path.join(TEST_IMAGE_FOLDER, image_file)
    label_path = os.path.join(TEST_LABEL_FOLDER, label_file)
    # 이미지 파일 열기
    with open(image_path, 'rb') as image:
        image_data = Image.open(image)
        image_data = image_data.convert('RGB')
    # 라벨 파일 열기
    with open(label_path, 'r') as label:
        label_data = json.load(label)
    # 이미지와 라벨 데이터 묶기
    data.append({
        'image':image_data,
        'text':label_data
    })
    
# Dataset으로 변환
if TEST_IMAGE_FOLDER[25:] == "KFashion":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['summary'] for item in data]
    })
elif TEST_IMAGE_FOLDER[25:] == "Nike" or TEST_IMAGE_FOLDER[25:] == "Zara":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [
            " | ".join([
                str(item['text']['Prompt']), 
                str(item['text']['Input']), 
                str(item['text']['Add_Info'])
            ])
            for item in data
        ]
    })

# Test DatasetDict 생성 및 전처리 적용
test_dataset = DatasetDict({'test': dataset})
test_dataset = test_dataset.with_transform(preprocess_data)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset['test'],
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=BATCH_SIZE
)
print(test_dataset)

DatasetDict({
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 200
    })
})


### Test Model

In [9]:
def calculate_fid(test_dataset, test_label_file, pipe):
    from torchvision import transforms as T
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fid_metric = FrechetInceptionDistance(feature=64).to(device)
    fid_metric.reset()
    
    # PIL 이미지를 uint8 텐서로 변환하는 transform 사용
    transform = T.PILToTensor()
    
    # 임시 폴더(생성 이미지 저장용) 생성
    tmp_folder = '../Data/Total/tmp'
    os.makedirs(tmp_folder, exist_ok=True)
    for idx, filename in enumerate(test_label_file):
        prompt = test_dataset['test'][idx]['text'][16:]
        generated_image = pipe(prompt).images[0]
        # generate image 저장   
        filename = filename.split('.')[0]
        generated_image.save(f'{tmp_folder}/{filename}.jpg')
    
        # real image
        real_image = test_dataset['test'][idx]['image']
        
        fake_tensor = transform(generated_image).unsqueeze(0).to(device)
        real_tensor = transform(real_image).unsqueeze(0).to(device)
        
        fid_metric.update(real_tensor, real=True)
        fid_metric.update(fake_tensor, real=False)
    
    fid_score = fid_metric.compute().item()
    return fid_score

fid_score = calculate_fid(test_dataset, TEST_LABEL_FILE, pipe)

In [ ]:
# 생성한 이미지와 원본 이미지 비교하여 precision과 recall 계산
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)],key=lambda x:int(x.split('.')[0]))

# TensorFlow session과 evaluator 초기화
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# 배치 생성기 준비 (원본 이미지)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# 배치 생성기 준비 (증강 이미지)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# 원본 이미지에 대한 activation 추출 (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# precision과 recall 계산
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])

# 폴더가 존재하면 삭제
shutil.rmtree(GENERATE_IMAGE_FOLDER)

In [14]:
print(f'Final test FID score: {fid_score:.5f}')
print(f'Final test precision: {precision:.5f}')
print(f'Final test recall: {recall:.5f}')

Final test FID score: 2.66617
Final test precision: 0.44500
Final test recall: 0.32500
